# Anima + WAI-Anima — ComfyUI Colab

Один ноутбук для Anima Aesthetic v1.1 и WAI-Anima v1.0. Устанавливает ComfyUI, ComfyUI-Manager, Anima-LLLite и готовые T2I/ControlNet/Inpaint workflow. Токены берутся из Colab Secrets (`HF_TOKEN`, `CIVITAI_API_TOKEN`) или запрашиваются интерактивно.

In [ ]:
# @title 1) Tokens and paths
import os, getpass
from pathlib import Path
try:
    from google.colab import userdata
except Exception:
    userdata = None
def secret(name):
    value = ''
    if userdata is not None:
        try: value = userdata.get(name) or ''
        except Exception: pass
    return value.strip() or os.environ.get(name, '').strip()
HF_TOKEN = secret('HF_TOKEN') or secret('HUGGINGFACE_TOKEN')
CIVITAI_API_TOKEN = secret('CIVITAI_API_TOKEN')
if not HF_TOKEN: HF_TOKEN = getpass.getpass('Hugging Face token (required): ').strip()
if not CIVITAI_API_TOKEN: CIVITAI_API_TOKEN = getpass.getpass('Civitai API token (recommended): ').strip()
if not HF_TOKEN: raise RuntimeError('HF_TOKEN is required.')
os.environ.update({'HF_TOKEN': HF_TOKEN, 'HUGGINGFACE_TOKEN': HF_TOKEN})
if CIVITAI_API_TOKEN: os.environ['CIVITAI_API_TOKEN'] = CIVITAI_API_TOKEN
COMFY_ROOT = Path('/content/ComfyUI'); MODEL_ROOT = COMFY_ROOT / 'models'
print('Tokens configured without displaying their values.')

In [ ]:
# @title 2) Install ComfyUI, Manager and Anima nodes
import subprocess
def run(cmd):
    print('+', cmd); subprocess.run(cmd, shell=True, check=True)
if not COMFY_ROOT.exists(): run('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI')
run('pip install -q -r /content/ComfyUI/requirements.txt')
NODES = {'ComfyUI-Manager':'https://github.com/ltdrdata/ComfyUI-Manager.git','ComfyUI-Anima-LLLite':'https://github.com/kohya-ss/ComfyUI-Anima-LLLite.git','comfyui_controlnet_aux':'https://github.com/Fannovel16/comfyui_controlnet_aux.git','comfyui-lora-manager':'https://github.com/willmiao/ComfyUI-Lora-Manager.git','rgthree-comfy':'https://github.com/rgthree/rgthree-comfy.git','was-node-suite-comfyui':'https://github.com/WASasquatch/was-node-suite-comfyui.git','ComfyUI-Image-Saver':'https://github.com/alexopus/ComfyUI-Image-Saver.git'}
for folder, repo in NODES.items():
    target = COMFY_ROOT/'custom_nodes'/folder
    if not target.exists(): run(f'git clone --depth 1 {repo} {target}')
    req = target/'requirements.txt'
    if req.exists(): run(f'pip install -q -r {req}')
print('ComfyUI and Anima node set are ready.')

In [ ]:
# @title 3) Download both Anima checkpoints and dependencies
import requests
def download(url, target, headers=None):
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>1024: print('exists:',target); return
    h={'Authorization':f'Bearer {HF_TOKEN}'} if 'huggingface.co' in url else {}
    if headers: h.update(headers)
    with requests.get(url,headers=h,stream=True,timeout=60) as r:
        r.raise_for_status()
        with open(target,'wb') as f:
            for chunk in r.iter_content(1024*1024):
                if chunk: f.write(chunk)
    print('downloaded:',target)
def civitai(url,target):
    h={'Authorization':f'Bearer {CIVITAI_API_TOKEN}'} if CIVITAI_API_TOKEN else {}
    download(url,target,h)
download('https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors',MODEL_ROOT/'text_encoders/qwen_3_06b_base.safetensors')
download('https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors',MODEL_ROOT/'vae/qwen_image_vae.safetensors')
download('https://huggingface.co/Comfy-Org/Anima-LLLite/resolve/main/model_patches/anima-lllite-any-test-like-v2.safetensors',MODEL_ROOT/'model_patches/anima-lllite-any-test-like-v2.safetensors')
download('https://huggingface.co/Comfy-Org/Anima-LLLite/resolve/main/model_patches/anima-lllite-inpainting-v2.safetensors',MODEL_ROOT/'model_patches/anima-lllite-inpainting-v2.safetensors')
civitai('https://civitai.red/api/download/models/3126581?fileId=3007030',MODEL_ROOT/'diffusion_models/anima/anima_aestheticV11.safetensors')
civitai('https://civitai.red/api/download/models/2983680?fileId=2863158',MODEL_ROOT/'diffusion_models/anima/waiANIMA_v10Base10.safetensors')
print('Both checkpoints are available in ComfyUI.')

In [ ]:
# @title 4) Install supplied v45 workflows as WAI and Aesthetic variants
import requests, json, urllib.parse
WORKFLOW_DIR=COMFY_ROOT/'user'/'default'/'workflows'/'anima'; WORKFLOW_DIR.mkdir(parents=True,exist_ok=True)
RAW='https://raw.githubusercontent.com/ekkonwork/ComfyUI-Workflow-Collection/main/workflows/anima/'
FILES={'Anima Workflow.json':'Anima Workflow','Anima Controlnet Workflow.json':'Anima Controlnet Workflow','Anima Inpaint Workflow.json':'Anima Inpaint Workflow'}
for source, stem in FILES.items():
    data=requests.get(urllib.parse.urljoin(RAW,urllib.parse.quote(source)),timeout=60).json()
    (WORKFLOW_DIR/(stem+' - WAI.json')).write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
    for node in data.get('nodes',[]):
        vals=node.get('widgets_values')
        if isinstance(vals,list): node['widgets_values']=[str(v).replace('waiANIMA_v10Base10.safetensors','anima_aestheticV11.safetensors') for v in vals]
    (WORKFLOW_DIR/(stem+' - Aesthetic.json')).write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
print('Workflow variants installed:',len(list(WORKFLOW_DIR.glob('*.json'))))

In [ ]:
# @title 5) Launch ComfyUI + Cloudflare tunnel
import subprocess,time,re
subprocess.run('apt-get -qq update && apt-get -qq install -y cloudflared',shell=True,check=True)
comfy=subprocess.Popen(['python','main.py','--listen','0.0.0.0','--port','8188'],cwd=COMFY_ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
time.sleep(8)
tunnel=subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:8188','--no-autoupdate'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
url=None; deadline=time.time()+90
while time.time()<deadline and url is None:
    line=tunnel.stdout.readline(); m=re.search(r'https://[-a-z0-9]+\.trycloudflare\.com',line,re.I)
    if m: url=m.group(0); print('Open ComfyUI:',url)
if not url: print('Tunnel URL was not detected; inspect output and restart the tunnel.')